In [ ]:
import pandas as pd
df = pd.read_csv(r'F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\game_tracking_2023.csv')

In [2]:
df.columns.tolist()

['game_id',
 'play_id',
 'nfl_id',
 'time',
 'x',
 'y',
 's',
 'a',
 'dis',
 'o',
 'dir',
 'event']

In [3]:
print("Shape:", df.shape)

display(df.head())

df.info(memory_usage="deep")

Shape: (3665443, 12)


,game_id,play_id,nfl_id,time,x,y,s,a,dis,o,dir,event
0,2024010707,570,56450,2024-01-07T21:46:07.700,42.17,25.29,0.0,0.0,0.0,290.97,213.26,NaN
1,2024010707,570,56450,2024-01-07T21:46:07.800,42.17,25.29,0.0,0.0,0.0,290.97,210.36,NaN
2,2024010707,570,56450,2024-01-07T21:46:07.900,42.18,25.29,0.0,0.0,0.0,290.97,205.45,NaN
3,2024010707,570,56450,2024-01-07T21:46:08.000,42.18,25.29,0.0,0.0,0.0,290.97,200.73,NaN
4,2024010707,570,56450,2024-01-07T21:46:08.100,42.18,25.29,0.0,0.0,0.0,290.97,199.27,NaN


<class 'pandas.DataFrame'>
RangeIndex: 3665443 entries, 0 to 3665442
Data columns (total 12 columns):
 #   Column   Dtype  
---  ------   -----  
 0   game_id  int64  
 1   play_id  int64  
 2   nfl_id   int64  
 3   time     str    
 4   x        float64
 5   y        float64
 6   s        float64
 7   a        float64
 8   dis      float64
 9   o        float64
 10  dir      float64
 11  event    str    
dtypes: float64(7), int64(3), str(2)
memory usage: 649.8 MB


In [4]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})

display(missing_summary)

,missing_count,missing_pct
game_id,0,0.00
play_id,0,0.00
nfl_id,0,0.00
time,0,0.00
x,0,0.00
y,0,0.00
s,0,0.00
a,0,0.00
dis,0,0.00
o,0,0.00


### Missing Values Analysis — game_tracking_2023

**Findings:**
- All identifier, timestamp, position, and movement columns have 0% missing values.
- The `event` column contains 3,422,066 missing values (93.36%).

**Interpretation:**
- Tracking data contains continuous frame-level observations.
- Events are recorded only at specific moments during a play.
- Missing `event` values are likely structural and may indicate frames without an annotated event.

**Data Quality Decision:**
- Preserve all tracking frames.
- Retain missing `event` values.
- Do not automatically drop rows or apply forward-fill imputation to `event`.

**Conclusion:**
The dataset is complete for identifiers and movement measurements. High missingness in `event` is consistent with event-based annotations in frame-level tracking data and is not necessarily a data quality issue.

In [6]:
print("Unique Games:", df["game_id"].nunique())

print("Unique Plays:",
      df[["game_id", "play_id"]].drop_duplicates().shape[0])

print("Unique Players:", df["nfl_id"].nunique())

display(df["event"].value_counts(dropna=False))

Unique Games: 333
Unique Plays: 31690
Unique Players: 156


event
NaN                          3422066
ball_snap                      50813
pass_forward                   31422
tackle                         29011
first_contact                  28345
pass_arrived                   23967
pass_outcome_caught            18845
handoff                        14053
pass_outcome_incomplete        10862
play_action                     8274
out_of_bounds                   6082
autoevent_passinterrupted       4237
run                             2899
qb_sack                         2104
man_in_motion                   1827
line_set                        1610
play_submit                     1264
touchdown                       1174
dropped_pass                     807
pass_outcome_interception        754
pass_outcome_touchdown           646
pass_tipped                      561
fumble                           496
qb_kneel                         493
qb_slide                         419
shift                            397
fumble_offense_recovered        

### Categorical Validation — game_tracking_2023

**Dataset Coverage**
- Unique games: 333
- Unique game-play combinations: 31,690
- Unique tracked players: 156

**Event Distribution**
- The most frequent non-null events include `ball_snap`, `pass_forward`, `tackle`, `first_contact`, and `pass_arrived`.
- Missing event annotations account for 93.36% of tracking records.
- Multiple tracking observations may share the same event within a play.

**Findings**
- Event categories appear consistent with NFL gameplay.
- Event frequencies represent tracking observations, not necessarily distinct plays.
- The dataset contains 333 games, compared with 334 games in the 2023 season of `games.csv`. The missing game ID should be checked before drawing conclusions about coverage.

**Conclusion**
The categorical distributions are generally plausible. No confirmed invalid event categories were identified in the displayed results.

In [7]:
candidate_key = ["game_id", "play_id", "nfl_id", "time"]

print("Full Duplicate Rows:", df.duplicated().sum())

print(
    "Duplicate Candidate Keys:",
    df.duplicated(subset=candidate_key).sum()
)

print(
    "Missing Values in Candidate Key:",
    df[candidate_key].isna().sum().sum()
)

Full Duplicate Rows: 0
Duplicate Candidate Keys: 0
Missing Values in Candidate Key: 0


### Granularity and Unique Identifier Validation

**Dataset:** `game_tracking_2023.csv`

**Granularity:** One tracking observation per player per timestamp within a game play.

**Validated Candidate Composite Key:**
`(game_id, play_id, nfl_id, time)`

**Validation Results:**
- Full duplicate rows: 0
- Duplicate candidate keys: 0
- Missing values in candidate key columns: 0

**Conclusion:**
The four-column composite key uniquely identifies each tracking observation in the current dataset. No duplicate records or null candidate keys were detected.

In [8]:
numeric_cols = ["x", "y", "s", "a", "dis", "o", "dir"]

display(df[numeric_cols].describe().T)

print("Negative Speed:", (df["s"] < 0).sum())
print("Negative Distance:", (df["dis"] < 0).sum())

,count,mean,std,min,25%,50%,75%,max
x,3665443.0,60.177652,24.513774,-5.89,40.98,60.29,79.34,126.73
y,3665443.0,26.793436,9.696727,-7.29,21.04,26.77,32.49,61.24
s,3665443.0,2.530810,2.127076,0.00,0.81,2.08,3.79,12.67
a,3665443.0,1.635531,1.293810,0.00,0.65,1.40,2.35,18.33
dis,3665443.0,0.256782,0.213518,0.00,0.08,0.21,0.38,3.44
o,3665443.0,177.683533,100.821942,0.00,88.21,177.50,268.08,360.00
dir,3665443.0,179.621000,103.361639,0.00,89.01,179.83,269.41,360.00


Negative Speed: 0
Negative Distance: 0


### Numeric Validation — game_tracking_2023

**Total Tracking Records:** 3,665,443

**Validation Results:**
- Negative speed (`s`) records: 0
- Negative distance (`dis`) records: 0
- Orientation (`o`) ranges from 0 to 360 degrees.
- Movement direction (`dir`) ranges from 0 to 360 degrees.

**Coordinate Ranges:**
- `x`: -5.89 to 126.73
- `y`: -7.29 to 61.24

Some coordinates fall outside standard NFL field dimensions. These values require interpretation according to the tracking coordinate system and should not automatically be treated as errors.

**Potential Extreme Values:**
- Maximum speed: 12.67
- Maximum acceleration: 18.33
- Maximum distance: 3.44

**Data Quality Decision:**
- Preserve all original tracking observations.
- Document extreme values for review during preprocessing.
- Validate coordinate conventions and measurement units before applying outlier treatment.

**Conclusion:**
No negative speed or distance values were detected. Orientation and movement direction fall within the expected angular range. Potential coordinate and movement extremes have been documented.

In [11]:
games_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\games.csv")
players_df = pd.read_csv(r"F:\Rehab work\NFL Hackathon\Raw Data\nfl-big-data-bowl-2027\players.csv")

print(
    "Tracking Game IDs missing from games:",
    len(set(df["game_id"]) - set(games_df["game_id"]))
)

print(
    "Tracking Player IDs missing from players:",
    len(set(df["nfl_id"]) - set(players_df["nfl_id"]))
)

print(
    "2023 Games without tracking:",
    len(
        set(games_df.loc[games_df["season"] == 2023, "game_id"])
        - set(df["game_id"])
    )
)

Tracking Game IDs missing from games: 0
Tracking Player IDs missing from players: 0
2023 Games without tracking: 1


In [12]:
missing_games = (
    set(games_df.loc[games_df["season"] == 2023, "game_id"])
    - set(df["game_id"])
)

print(missing_games)

{2023100108}


### Cross-Table Coverage Finding — 2023

- **Missing Game ID:** `2023100108`
- The game exists in `games.csv`.
- The game is absent from both `player_play.csv` and `game_tracking_2023.csv`.
- This confirms a shared coverage gap across the two player-level datasets.

**Preprocessing Decision:**
- Retain the game in `games.csv`.
- Document the missing coverage.
- Account for this game when joining or comparing datasets.

**Status:** Confirmed coverage gap; underlying cause unknown.